# A simple 2D Discrete Flow Matching model

This notebook trains and evaluates a simple 2D discrete FM model with $\kappa_t = t^2$ scheduler.

Dataset: Manually defined code segments, 
Model (probability denoiser): MLP

## Imports and init device

In [2]:
import time
import torch

from torch import nn, Tensor

# flow_matching
from flow_matching.path import MixtureDiscreteProbPath
from flow_matching.path.scheduler import PolynomialConvexScheduler
from flow_matching.solver import MixtureDiscreteEulerSolver
from flow_matching.utils import ModelWrapper
from flow_matching.loss import MixturePathGeneralizedKL

# visualization
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt

In [3]:
if torch.cuda.is_available(): # Checks which device to use
    device = 'cuda:0'
    print('Using gpu')
else:
    device = 'cpu'
    print('Using cpu.')

Using gpu


In [4]:
torch.manual_seed(42)

## Dataset

I had to make some changes to the inital dataset that way we could generate code/text instead of using the 2D toy version.
In this implementation there is a sample dataset with python code segments

In [13]:
import torch

# Sample dataset that contains simple python code segments
texts = [
    "def add(a, b): return a + b",
    "def sub(a, b): return a - b",
    "def mul(a, b): return a * b",
    "for i in range(10): print(i)",
    "if x > 0:\n    print('positive')",
    "while n > 1:\n    n -= 1",
    "class Counter:\n    def __init__(self):\n        self.x = 0",
    "def square(x): return x * x",
    "def div(a, b): return a / b if b != 0 else None",
    "def mod(a, b): return a % b",
    "for j in range(5):\n    print(j * j)",
    "try:\n    value = int('42')\nexcept ValueError:\n    value = 0",
    "with open('file.txt', 'w') as f:\n    f.write('hello')",
    "items = [1, 2, 3]\nitems.append(4)",
    "data = {'x': 1, 'y': 2}\nprint(data.get('x'))",
    "def factorial(n):\n    return 1 if n <= 1 else n * factorial(n - 1)",
    "name = 'flow'\nprint(name.upper())",
    "result = sum([1, 2, 3, 4])"
]

all_text = "".join(texts) # Get all characters in datase
chars = sorted(list(set(all_text))) # Sort characters and remove duplicates to create d

# Reserve token 0 for padding
stoi = {ch: i + 1 for i, ch in enumerate(chars)} # Numeric (index) value for a string
itos = {i + 1: ch for i, ch in enumerate(chars)} # String value for an index

PAD_TOKEN = 0 # Used in the event the sequence length needs to be matched but text sequence is too short
vocab_size = len(chars) + 1   # +1 because 0 is padding
seq_len = 64 # Our N in DFM


# Converts strings into token IDS
def encode_text(s, seq_len=64):
    ids = [stoi[c] for c in s if c in stoi] # Converts to numerical value
    ids = ids[:seq_len] # Only keeps first 64 tokens
    if len(ids) < seq_len: # Pads with more tokens if necessary
        ids += [PAD_TOKEN] * (seq_len - len(ids))
    return ids

#Encodes every text in the dataset to be a list of 64 ints
encoded_dataset = torch.tensor(
    [encode_text(t, seq_len) for t in texts],
    dtype=torch.long
)


def inf_train_gen(batch_size=32, device="cpu"):
    idx = torch.randint(0, encoded_dataset.shape[0], (batch_size,)) #Samples random rows from dataset
    x_end = encoded_dataset[idx].to(device) # sends data to devic
    return x_end.long() 

print("Dataset shape:", encoded_dataset.shape)
print("Vocab size:", vocab_size)
print("Sequence length:", seq_len)

Dataset shape: torch.Size([18, 64])
Vocab size: 57
Sequence length: 64


## Model

In [14]:
# Activation class
class Swish(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x: Tensor) -> Tensor: 
        return torch.sigmoid(x) * x

# Model class
class MLP(nn.Module):
    def __init__(
        self, input_dim: int = 128, time_dim: int = 1, hidden_dim=128, length=2):
        super().__init__()
        self.input_dim = input_dim
        self.time_dim = time_dim
        self.hidden_dim = hidden_dim

        self.time_embedding = nn.Linear(1, time_dim)
        self.token_embedding = torch.nn.Embedding(self.input_dim, hidden_dim)

        self.main = nn.Sequential(
            Swish(),
            nn.Linear(hidden_dim * length + time_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, self.input_dim * length),
        )

    def forward(self, x, t):
        t = self.time_embedding(t.unsqueeze(-1))
        x = self.token_embedding(x)

        B, N, d = x.shape
        x = x.reshape(B, N * d)
        
        h = torch.cat([x, t], dim=1)
        h = self.main(h)

        h = h.reshape(B, N, self.input_dim)

        return h

## Train Discrete Flow Matching model with a uniform source distribution

In [ ]:
source_distribution = "uniform" #Setting to uniform so all tokens randomly chosen

# training arguments
lr = 0.001
batch_size = 4096
iterations = 30001
print_every = 3000

vocab_size = 128
hidden_dim = 128

epsilon = 1e-3

if source_distribution == "uniform":
    added_token = 0
elif source_distribution == "mask":
    mask_token = 0 #vocab_size # tokens starting from zero
    added_token = 1
else:
    raise NotImplementedError
    
# additional mask token
vocab_size += added_token

# probability denoiser model init
probability_denoiser = MLP(input_dim=vocab_size, time_dim=1, hidden_dim=hidden_dim, length=seq_len).to(device) # Creating Neural Network

# instantiate a convex path object
scheduler = PolynomialConvexScheduler(n=2.0) # How quickly we move from x_0 to x_1
path = MixtureDiscreteProbPath(scheduler=scheduler) 

# init optimizer
optim = torch.optim.Adam(probability_denoiser.parameters(), lr=lr) 

loss_fn = MixturePathGeneralizedKL(path=path)

# train
start_time = time.time()

steps = 0
losses = []
for i in range(iterations):
    optim.zero_grad() # Clear previous gradients

    # sample data (user's responsibility): in this case, (X_0,X_1) ~ pi(X_0,X_1)
    x_1 = inf_train_gen(batch_size=batch_size, device=device) # sample data
    
    if source_distribution == "uniform":
        x_0 = torch.randint_like(x_1, high=vocab_size)
    elif source_distribution == "mask":
        x_0 = torch.zeros_like(x_1) + mask_token
    else:
        raise NotImplementedError

    # sample time (user's responsibility)
    t = torch.rand(x_1.shape[0]).to(device) * (1 - epsilon)

    # sample probability path
    path_sample = path.sample(t=t, x_0=x_0, x_1=x_1)# sample intermediate sequence

    # discrete flow matching generalized KL loss
    logits = probability_denoiser(x=path_sample.x_t, t=path_sample.t) # transition probabilities predicted by the model for each token
    loss = loss_fn(logits=logits, x_1=x_1, x_t=path_sample.x_t, t=path_sample.t)

    # optimizer step
    loss.backward() # backward
    optim.step() # update
    
    # log loss
    if (i+1) % print_every == 0:
        elapsed = time.time() - start_time
        print('| iter {:6d} | {:5.2f} ms/step | loss {:8.3f} ' 
              .format(i+1, elapsed*1000/print_every, loss.item())) 
        start_time = time.time()

| iter   3000 | 42.52 ms/step | loss    0.089 
| iter   6000 | 49.03 ms/step | loss    0.090 
| iter   9000 | 48.84 ms/step | loss    0.081 
| iter  12000 | 52.17 ms/step | loss    0.081 
| iter  15000 | 46.32 ms/step | loss    0.084 
| iter  18000 | 52.91 ms/step | loss    0.084 
| iter  21000 | 47.10 ms/step | loss    0.086 
| iter  24000 | 50.97 ms/step | loss    0.081 
| iter  27000 | 44.77 ms/step | loss    0.080 
| iter  30000 | 56.69 ms/step | loss    0.077 


#### Sample from trained model

In [ ]:
class WrappedModel(ModelWrapper): # Assigns probabilities to the model outputs
    def forward(self, x: torch.Tensor, t: torch.Tensor, **extras):
        return torch.softmax(self.model(x, t, **extras), dim=-1)

wrapped_probability_denoiser = WrappedModel(probability_denoiser)
solver = MixtureDiscreteEulerSolver(model=wrapped_probability_denoiser, path=path, vocabulary_size=vocab_size)

## Generating New Samples

In [ ]:

n_samples = 15

# Samples the initial sequence
if source_distribution == "uniform":
    x_init = torch.randint(
        low=0,
        high=vocab_size,
        size=(n_samples, seq_len),
        device=device
    )
elif source_distribution == "mask":
    x_init = (torch.zeros(size=(n_samples, seq_len), device=device) + PAD_TOKEN).long()
else:
    raise ValueError(f"Unknown source_distribution: {source_distribution}")

# Generates samples from the mode
with torch.no_grad():
    samples = solver.sample(
        x_init=x_init,
        step_size=1 / 64,
        verbose=True,
        return_intermediates=False,
    )

# Used to convert the ids back to a character
def decode(ids):
    chars_out = []
    for i in ids:
        i = int(i)
        if i == PAD_TOKEN:
            continue
        if i in itos:
            chars_out.append(itos[i])
    return "".join(chars_out)

print("\nGenerated samples:\n")
for i, sample in enumerate(samples.cpu()):
    print(f"Sample{i+1}:")
    print(decode(sample))
    print()

NFE: 64: 100%|██████████| 1.0/1.0 [00:03<00:00,  3.93s/it]     


Generated samples:

Sample1:
for i in range(10): print(i)

Sample2:
def add(a, b): return a + b

Sample3:
try:
    value = int('42')
except ValueError:
    value = 0

Sample4:
items = [1, 2, 3]
items.append(4)

Sample5:
for i in range(10): print(i)

Sample6:
try:
    value = int('42')
except ValueError:
    value = 0

Sample7:
def add(a, b): return a + b

Sample8:
data = {'x': 1, 'y': 2}
print(data.get('x'))

Sample9:
def add(a, b): return a + b

Sample10:
def square(x): return x * x

Sample11:
with open('file.txt', 'w') as f:
    f.write('hello')

Sample12:
def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 

Sample13:
def square(x): return x * x

Sample14:
data = {'x': 1, 'y': 2}
print(data.get('x'))

Sample15:
name = 'flow'
print(name.upper())



The samples turned out pretty well, but that may be due to the little complexty in the actual training data. As we can see in Sample 12 the output got cut off because we had a sequnece length of 64 and some of the other outputs are repeated as well. As far as implementing DFM, the model did a great job on outputs and trained at a decent time.